# 03. R4T 심화: 조건부 denoising과 단일 fan-out 검색

> 이 노트북은 R4T의 3단계를 설명하기 위한 **toy reproduction**입니다. 실제 VE/EDM Diffusion Transformer, classifier-free guidance, SDE solver, Polyvore·음악 임베딩을 재현하지 않습니다. NumPy 선형 회귀로 만든 조건부 평균과 분석적 shrinkage denoiser만 사용하며 논문 성능·지연시간 재현이 아닙니다. 네트워크·API·GPU를 사용하지 않습니다.

## 학습 목표

1. 논문의 VE diffusion 목표 `Z_t = Z_target + sigma * epsilon`을 구현합니다.
2. query-conditioned toy denoiser가 noisy tensor의 MSE를 줄이는지 검증합니다.
3. `L`개 생성 방향을 한 행렬곱으로 데이터베이스에 매핑합니다.
4. R4T-FOLM과 R4T-Diffusion의 지연시간 주장을 올바른 경계 안에서 해석합니다.

In [1]:
import numpy as np

SEED = 260306397
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

def normalize(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

L, D = 4, 4
print(f'deterministic seed={SEED}, target tensor=(L={L}, d={D})')

deterministic seed=260306397, target tensor=(L=4, d=4)


## 조건부 목표 분포의 작은 대리 모델

논문의 diffusion model은 broad query embedding `z_q`를 조건으로 `Z_target in R^(L x d)`의 분포를 학습합니다. 아래에서는 서로 다른 검색 facet을 나타내는 네 선형 변환으로 합성 목표를 만들고, 질의에서 각 목표 행을 예측하는 ridge 회귀를 학습합니다. 이 회귀 결과 `mu(z_q)`는 Transformer가 학습해야 할 조건부 구조를 아주 거칠게 대신합니다.

In [2]:
facet_transforms = np.array([
    [[1.00, 0.00, 0.00, 0.00], [0.00, 1.00, 0.00, 0.00], [0.00, 0.00, 1.00, 0.00], [0.00, 0.00, 0.00, 1.00]],
    [[0.95, 0.20, 0.00, 0.00], [-0.15, 0.95, 0.10, 0.00], [0.00, 0.10, 0.95, 0.10], [0.00, 0.00, -0.10, 1.00]],
    [[0.90, 0.00, 0.30, 0.00], [0.00, 1.00, 0.00, 0.10], [-0.25, 0.00, 0.95, 0.00], [0.00, -0.10, 0.00, 1.00]],
    [[0.90, -0.25, 0.00, 0.10], [0.25, 0.90, 0.00, 0.00], [0.00, 0.00, 1.00, 0.15], [-0.10, 0.00, -0.10, 0.95]],
])

def make_clean_target(query):
    # 각 변환이 동일 broad intent에서 조금 다른 검색 방향을 만듭니다.
    return normalize(np.einsum('lij,j->li', facet_transforms, normalize(query)))

n_train = 96
train_queries = normalize(rng.normal(size=(n_train, D)) + np.array([0.8, 0.0, 0.2, 0.0]))
train_targets = np.stack([make_clean_target(q) for q in train_queries])

# 각 slot에 대해 q @ W_l ~= Z_target[l]인 작은 조건부 평균 모델을 학습합니다.
ridge = 1e-4
gram = train_queries.T @ train_queries + ridge * np.eye(D)
weights = np.stack([
    np.linalg.solve(gram, train_queries.T @ train_targets[:, slot, :])
    for slot in range(L)
])

def conditional_mean(query):
    q = normalize(query)
    return normalize(np.stack([q @ weights[slot] for slot in range(L)]))

held_out_query = normalize(np.array([0.90, 0.20, 0.35, 0.10]))
held_out_target = make_clean_target(held_out_query)
mean_prediction = conditional_mean(held_out_query)
mean_mse = float(np.mean((mean_prediction - held_out_target) ** 2))
print('conditional mean MSE:', f'{mean_mse:.8f}')
assert mean_prediction.shape == (L, D)
assert mean_mse < 1e-4

conditional mean MSE: 0.00004421


## VE noise와 toy denoiser

논문 식 (9)의 forward corruption과 가중 손실은 다음 형태입니다.

- `Z_t = Z_target + sigma * epsilon`, `epsilon ~ N(0, I)`
- `L_diff = E[lambda(sigma) * ||D_phi(Z_t; sigma, z_q) - Z_target||^2]`
- `lambda(sigma) = (sigma^2 + sigma_data^2) / (sigma * sigma_data)^2`

여기서는 학습된 neural score가 아니라 noisy 관측과 조건부 평균을 noise level에 따라 섞는 Gaussian shrinkage를 사용합니다. 따라서 아래 결과는 diffusion 모델 품질이 아니라 denoising 원리만 확인합니다.

In [3]:
sigma_data = 0.30
sigma = 0.80
epsilon = rng.normal(size=held_out_target.shape)
z_noisy = held_out_target + sigma * epsilon

def toy_denoiser(z_t, noise_sigma, query):
    # noise가 클수록 조건부 평균을 더 신뢰하고, 작을수록 관측을 더 보존합니다.
    observation_weight = sigma_data**2 / (sigma_data**2 + noise_sigma**2)
    return observation_weight * z_t + (1.0 - observation_weight) * conditional_mean(query)

z_denoised = toy_denoiser(z_noisy, sigma, held_out_query)
noisy_mse = float(np.mean((z_noisy - held_out_target) ** 2))
denoised_mse = float(np.mean((z_denoised - held_out_target) ** 2))
edm_weight = (sigma**2 + sigma_data**2) / (sigma * sigma_data) ** 2
weighted_loss = edm_weight * np.sum((z_denoised - held_out_target) ** 2)

print(f'noisy MSE={noisy_mse:.6f}')
print(f'denoised MSE={denoised_mse:.6f}')
print(f'lambda(sigma)={edm_weight:.4f}, weighted squared error={weighted_loss:.6f}')
assert denoised_mse < noisy_mse
assert np.isfinite(weighted_loss) and weighted_loss >= 0.0
print('Assertions passed: toy conditional denoiser가 오차를 줄였습니다.')

noisy MSE=0.789392
denoised MSE=0.012633
lambda(sigma)=12.6736, weighted squared error=2.561718
Assertions passed: toy conditional denoiser가 오차를 줄였습니다.


## `L`개 방향을 함께 생성하고 한 번에 검색하기

논문의 R4T-Diffusion은 `L`개 임베딩을 하나의 coherent tensor로 생성한 다음 각 행을 최근접 데이터베이스 콘텐츠로 매핑합니다. 아래 toy sampler는 high-noise tensor에서 시작해 네 noise level을 거치며 같은 query-conditioned denoiser를 적용합니다. 이는 논문의 256-step probability-flow SDE solver를 재현하지 않습니다. 마지막 검색은 `generated @ database.T`라는 한 번의 배치 행렬곱입니다.

In [4]:
deployment_query = normalize(np.array([0.75, 0.15, 0.55, 0.05]))
generated = rng.normal(size=(L, D))
noise_schedule = [1.0, 0.5, 0.2, 0.05]
for current_sigma in noise_schedule:
    generated = toy_denoiser(generated, current_sigma, deployment_query)
generated = normalize(generated)

# 검색 DB: 조건 목표 근처의 항목 4개와 distractor 60개를 만듭니다.
expected_facets = make_clean_target(deployment_query)
nearby_items = normalize(expected_facets + rng.normal(0.0, 0.02, size=(L, D)))
distractors = normalize(rng.normal(size=(60, D)))
retrieval_database = np.vstack([nearby_items, distractors])
item_names = np.array([f'facet_{i}' for i in range(L)] + [f'distractor_{i}' for i in range(60)])

# batch 방식: L개 방향을 한 행렬곱에 넣습니다.
similarities = generated @ retrieval_database.T
batch_ids = np.argmax(similarities, axis=1)

# 의미 비교를 위한 순차 방식: 방향마다 똑같은 최근접 이웃 계산을 반복합니다.
sequential_ids = np.array([np.argmax(direction @ retrieval_database.T) for direction in generated])

print('generated tensor:', generated.shape)
print('retrieved items :', item_names[batch_ids].tolist())
print('batch == sequential:', bool(np.array_equal(batch_ids, sequential_ids)))
assert generated.shape == (L, D)
assert similarities.shape == (L, len(retrieval_database))
assert np.array_equal(batch_ids, sequential_ids)
assert np.all(batch_ids < L)  # 이 seed에서는 네 의도 facet 주변 항목을 모두 찾습니다.

generated tensor: (4, 4)
retrieved items : ['facet_0', 'facet_1', 'facet_2', 'facet_3']
batch == sequential: True


## 비교와 지연시간 해석

| 배치 방식 | fan-out 생성 | DB 검색 | 핵심 비용 |
|---|---|---|---|
| R4T-FOLM | RL-tuned LLM이 `k`개 하위 질의를 autoregressive 생성 | 각 하위 질의 검색(구현에 따라 batching 가능) | token generation과 반복 검색 |
| Best-of-N | fan-out을 `N`번 생성한 뒤 보상으로 선택 | 후보마다 검색 | 높은 품질과 큰 online 비용의 교환 |
| R4T-Diffusion | `L`개 방향을 하나의 tensor로 공동 샘플링 | 한 batch nearest-neighbor 연산 | diffusion solver와 ANN index |

논문에서 ‘single-pass fan-out’은 **결과 집합 전체를 공동 생성한다**는 뜻이지 denoiser를 정확히 한 번 호출한다는 뜻이 아닙니다. 논문 구현은 256 SDE solver steps를 사용합니다. 또한 batch 행렬곱과 ANN index의 실제 지연시간은 DB 크기, 하드웨어, batching, 메모리 이동, solver 구현에 좌우됩니다. 이 작은 NumPy 예제의 실행 시간을 논문의 ‘한 자릿수 배 이상’ 개선 주장과 비교하면 안 됩니다.

In [5]:
# 시간을 재는 대신 비교 가능한 호출 구조만 명시합니다. 실제 latency benchmark가 아닙니다.
cost_model = {
    'R4T-FOLM': {'joint_output_tensor': False, 'autoregressive_fanout': True, 'retrieval_batches_min': 1},
    'R4T-Diffusion': {'joint_output_tensor': True, 'denoising_steps_in_paper': 256, 'retrieval_batches': 1},
}
for method, properties in cost_model.items():
    print(method, '->', properties)
print('Caveat: 호출 수는 wall-clock latency가 아니며 이 toy는 논문의 속도 수치를 재현하지 않습니다.')

R4T-FOLM -> {'joint_output_tensor': False, 'autoregressive_fanout': True, 'retrieval_batches_min': 1}
R4T-Diffusion -> {'joint_output_tensor': True, 'denoising_steps_in_paper': 256, 'retrieval_batches': 1}
Caveat: 호출 수는 wall-clock latency가 아니며 이 toy는 논문의 속도 수치를 재현하지 않습니다.


## 예상 출력과 한계

조건부 평균 MSE는 매우 작고, denoised MSE는 noisy MSE보다 작아야 합니다. 생성 텐서 크기는 `(4, 4)`, 배치와 순차 검색 ID는 같고 각 assertion이 통과해야 합니다.

- 선형 조건부 평균은 다봉성 분포를 표현하지 못하며 평균 붕괴를 일으킬 수 있습니다.
- toy denoiser는 학습된 score/velocity model이 아니고, 역 SDE/ODE의 수치 정확성도 다루지 않습니다.
- 실제 R4T는 `L=12`, `d=128`, coherent Transformer, VE/EDM weighting, classifier-free guidance를 사용합니다.
- 정확한 최근접 검색을 썼습니다. 대규모 서비스에서는 ANN index의 recall-latency-memory trade-off를 별도로 평가해야 합니다.
- RL 합성 데이터의 편향·reward hacking은 diffusion 모델에 그대로 컴파일될 수 있으므로 데이터 감사가 필요합니다.
- 재현 연구에서는 공개 코드·데이터·체크포인트가 있는지 확인하고, 없는 구성은 논문 결과와 분리해 toy 실험으로 보고해야 합니다.